# CayleyPy + MultiGPUBeamSearch
РќР°Р¶РјРёС‚Рµ **Copy & Edit в†’ Run All**. РњРµРЅСЏР№С‚Рµ РіСЂР°С„, СЃС‚Р°СЂС‚, `beam_width` Рё `num_gpus`. РџРµСЂРІС‹Р№ Р·Р°РїСѓСЃРє СЃРѕР±РёСЂР°РµС‚ СЏРґСЂРѕ Рё Р·Р°РЅРёРјР°РµС‚ РЅРµСЃРєРѕР»СЊРєРѕ РјРёРЅСѓС‚.

`backend="native"` вЂ” MultiGPUBeamSearch, `backend="torch"` вЂ” CayleyPy. РћС†РµРЅРєР° РҐСЌРјРјРёРЅРіР° Рё РІС‹Р±РѕСЂ LibTorch/CUTLASS Р°РІС‚РѕРјР°С‚РёС‡РµСЃРєРёРµ. РџРѕРґРґРµСЂР¶РёРІР°СЋС‚СЃСЏ РїРµСЂРµСЃС‚Р°РЅРѕРІРѕС‡РЅС‹Рµ РіСЂР°С„С‹ СЃРѕ СЃРѕСЃС‚РѕСЏРЅРёРµРј РґРѕ 120 СЌР»РµРјРµРЅС‚РѕРІ.

Автоподбор учитывает выбранный луч и GPU. `beam_width="max"` резервирует память под быстрый батч и подбирает вместимый луч. Для нейронной оценки можно подключить поддерживаемую модель или бленд.


In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/TryDotAtwo/MultiGPUBeamSearch.git@fe31e073a7d990e46fa016aa5cd09d7d5a9b9db1#subdirectory=integrations/cayleypy_native"], check=True)


In [ ]:
import cayleypy
import multigpubeamsearch


In [ ]:
graph = cayleypy.CayleyGraph(cayleypy.PermutationGroups.lrx(4), device="cuda")
start = [3, 2, 1, 0]


In [ ]:
multigpubeamsearch.enable_native(multigpubeamsearch.NativeOptions(num_gpus=2))

result = graph.beam_search(
    start_state=start,
    beam_width=4096,
    max_steps=16,
    return_path=True,
    backend="native",  # "torch" вЂ” РїРѕРёСЃРє CayleyPy
)


In [ ]:
print("РџСѓС‚СЊ РЅР°Р№РґРµРЅ:", result.path_found)
if result.path_found:
    print("Р§РёСЃР»Рѕ С…РѕРґРѕРІ:", result.path_length)
    print(result.get_path_as_string())


In [ ]:
# РЎРѕС…СЂР°РЅРµРЅРёРµ СЂРµР·СѓР»СЊС‚Р°С‚Р° Рё Р»РѕРіРѕРІ; РґР»СЏ РёСЃРїРѕР»СЊР·РѕРІР°РЅРёСЏ РїРѕРёСЃРєР° РјРµРЅСЏС‚СЊ РЅРµ РЅСѓР¶РЅРѕ.
import json, shutil
from pathlib import Path

output = Path("/kaggle/working/beam-results")
output.mkdir(exist_ok=True)
metadata = getattr(result, "native_metadata", {})
(output / "result.json").write_text(json.dumps({
    "path_found": bool(result.path_found), "path": result.path,
    "path_length": result.path_length, "backend": getattr(result, "backend", "torch"),
    "native_metadata": metadata}, indent=2, default=str))
if metadata.get("run_dir"):
    run = Path(metadata["run_dir"])
    for p in run.rglob("*"):
        if p.is_file() and p.suffix in {".log", ".json", ".csv", ".txt"} and p.stat().st_size < 20_000_000:
            target = output / "logs" / p.relative_to(run)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, target)
